# Notebook 06 — Calibration, Thresholds & Error Analysis

Loads the best DiT checkpoint and performs full post-hoc analysis:

1. Temperature calibration before/after comparison
2. Threshold sensitivity analysis
3. Full test-set metrics
4. Confusion matrix, ROC and PR curves
5. Per-institution breakdown
6. Error analysis — false-safe deep-dive
7. Production readiness checklist


In [1]:
%matplotlib inline

import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from scipy.special import expit as sigmoid

from src.data.dataset import HallucinationRiskDataset
from src.models.dit_classifier import DiTClassifier
from src.models.calibrator import TemperatureCalibrator
from src.utils.metrics import compute_metrics, compute_per_institution_metrics, compute_ece
from src.utils.visualization import (
    plot_confusion_matrix, plot_roc_curve, plot_pr_curve,
    plot_calibration_curve, plot_per_institution_metrics, plot_score_distribution
)

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT           = Path('..').resolve()
METADATA_CSV   = ROOT / 'data' / 'metadata.csv'
RENDERED_DIR   = ROOT / 'data' / 'rendered_pages'
DIT_CKPT_DIR   = ROOT / 'checkpoints' / 'dit'
LOG_DIR        = ROOT / 'logs' / 'calibration_eval'
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ── Device ─────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

/Users/idan/.pyenv/versions/3.13.11/envs/doc-risk-classifier/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


## 1 — Load Best DiT Model and Run Inference


In [2]:
# ── Load checkpoint ──────────────────────────────────────────────────────────
ckpt_path = DIT_CKPT_DIR / 'dit_best.pt'
assert ckpt_path.exists(), f'Checkpoint not found: {ckpt_path}. Run notebook 05 first.'

ckpt = torch.load(ckpt_path, map_location=device)
model_name = ckpt.get('model_name', 'microsoft/dit-base')

dit = DiTClassifier(model_name=model_name, num_classes=1)
dit.load_state_dict(ckpt['model_state_dict'])
dit = dit.to(device)
dit.eval()
print(f'Loaded DiT from {ckpt_path}')
print(f'Saved thresholds: T_low={ckpt["thresholds"]["T_low"]:.3f}  T_high={ckpt["thresholds"]["T_high"]:.3f}')

# ── Datasets ─────────────────────────────────────────────────────────────────
val_ds  = HallucinationRiskDataset(METADATA_CSV, 'val',  RENDERED_DIR, augment=False)
test_ds = HallucinationRiskDataset(METADATA_CSV, 'test', RENDERED_DIR, augment=False)

val_loader  = DataLoader(val_ds,  batch_size=16, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=0)

# ── Inference helper ──────────────────────────────────────────────────────────
@torch.no_grad()
def collect_logits(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        logits = model(images).squeeze(-1)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels.squeeze(-1).numpy())
    return np.concatenate(all_logits), np.concatenate(all_labels)

print('Running inference on val set ...')
val_logits,  val_labels  = collect_logits(dit, val_loader,  device)
print('Running inference on test set ...')
test_logits, test_labels = collect_logits(dit, test_loader, device)

# ── Load metadata for error analysis ─────────────────────────────────────────
meta_df   = pd.read_csv(METADATA_CSV)
test_meta = meta_df[meta_df['split'] == 'test'].reset_index(drop=True) if 'split' in meta_df.columns else meta_df.iloc[:len(test_ds)].reset_index(drop=True)

print(f'Val  samples: {len(val_logits)}')
print(f'Test samples: {len(test_logits)}')

AssertionError: Checkpoint not found: /Users/idan/projects/sandbox/for_tal/checkpoints/dit/dit_best.pt. Run notebook 05 first.

## 2 — Calibration Analysis


In [ ]:
# ── Fit calibrator on val set ────────────────────────────────────────────────
calibrator = TemperatureCalibrator()
calibrator.calibrate(val_logits, val_labels)
print(f'Fitted temperature: {calibrator.temperature:.4f}')

# ── Pre- and post-calibration probs ──────────────────────────────────────────
val_probs_uncal = sigmoid(val_logits)                  # before scaling
val_probs_cal   = calibrator.predict(val_logits)       # after scaling

ece_before = compute_ece(val_labels.astype(int), val_probs_uncal)
ece_after  = compute_ece(val_labels.astype(int), val_probs_cal)
print(f'ECE before calibration: {ece_before:.4f}')
print(f'ECE after  calibration: {ece_after:.4f}')

# ── Plot calibration curve before vs after ────────────────────────────────────
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect')

if len(np.unique(val_labels)) >= 2:
    fop_uc, mpv_uc = calibration_curve(val_labels.astype(int), val_probs_uncal, n_bins=10, strategy='uniform')
    ax.plot(mpv_uc, fop_uc, 's-', lw=2, label=f'Before T-scaling (ECE={ece_before:.3f})', color='tomato')

    fop_c, mpv_c = calibration_curve(val_labels.astype(int), val_probs_cal, n_bins=10, strategy='uniform')
    ax.plot(mpv_c, fop_c, 's-', lw=2, label=f'After T-scaling  (ECE={ece_after:.3f})', color='steelblue')

ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curve — DiT')
ax.legend()
plt.tight_layout()
plt.show()

## 3 — Threshold Selection


In [ ]:
# ── Derive thresholds ────────────────────────────────────────────────────────
thresholds = calibrator.get_thresholds(
    val_probs_cal, val_labels, target_false_safe_rate=0.05
)
T_LOW  = thresholds['T_low']
T_HIGH = thresholds['T_high']
print(f'T_low = {T_LOW:.4f}  |  T_high = {T_HIGH:.4f}')

# ── Threshold sensitivity: false_safe_rate vs T_low candidates ───────────────
risky_mask   = val_labels.astype(int) == 1
n_risky      = risky_mask.sum()
candidates   = np.linspace(0.0, 1.0, 200)
fsr_curve    = []
review_curve = []

for t in candidates:
    pred_safe    = val_probs_cal < t
    fsr = pred_safe[risky_mask].mean() if n_risky > 0 else 0.0
    t_h = t + (1.0 - t) / 2.0
    review_rate = ((val_probs_cal >= t) & (val_probs_cal <= t_h)).mean()
    fsr_curve.append(fsr)
    review_curve.append(review_rate)

fig, ax1 = plt.subplots(figsize=(9, 5))
color_fsr    = 'tomato'
color_review = 'steelblue'

ax1.plot(candidates, fsr_curve, color=color_fsr, lw=2, label='False-safe rate')
ax1.axhline(0.05, color=color_fsr, linestyle='--', lw=1, label='Target 5%')
ax1.axvline(T_LOW, color='black', linestyle=':', lw=1.5, label=f'Selected T_low={T_LOW:.3f}')
ax1.set_xlabel('T_low threshold')
ax1.set_ylabel('False-safe rate', color=color_fsr)
ax1.tick_params(axis='y', labelcolor=color_fsr)

ax2 = ax1.twinx()
ax2.plot(candidates, review_curve, color=color_review, lw=2, label='Review rate')
ax2.axhline(0.30, color=color_review, linestyle='--', lw=1, label='Operational limit 30%')
ax2.set_ylabel('Review rate', color=color_review)
ax2.tick_params(axis='y', labelcolor=color_review)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
ax1.set_title('Threshold Sensitivity — False-Safe Rate vs T_low')
plt.tight_layout()
plt.show()

## 4 — Full Test-Set Metrics


In [ ]:
# ── Calibrate test probs ─────────────────────────────────────────────────────
test_probs = calibrator.predict(test_logits)

# ── Compute all metrics ──────────────────────────────────────────────────────
test_metrics = compute_metrics(test_labels.astype(int), test_probs, thresholds)

print('Test-Set Metrics')
print('=' * 40)
metrics_display = [
    ('F1 Score',              'f1'),
    ('Precision (safe)',       'precision_safe'),
    ('Recall (risky)',         'recall_risky'),
    ('False-safe Rate',        'false_safe_rate'),
    ('Review Rate',            'review_rate'),
    ('ROC AUC',                'roc_auc'),
    ('PR AUC',                 'pr_auc'),
    ('ECE (calibrated)',       'ece'),
]
for label, key in metrics_display:
    val = test_metrics[key]
    flag = ''
    if key == 'false_safe_rate' and not np.isnan(val) and val > 0.05:
        flag = ' ⚠ EXCEEDS TARGET'
    if key == 'ece' and not np.isnan(val) and val > 0.10:
        flag = ' ⚠ EXCEEDS TARGET'
    print(f'  {label:28s}: {val:.4f}{flag}')

# ── Ternary prediction array ─────────────────────────────────────────────────
t_low, t_high = thresholds['T_low'], thresholds['T_high']
test_preds = np.where(test_probs < t_low, 0, np.where(test_probs > t_high, 1, 2)).astype(int)

## 5 — Confusion Matrix


In [ ]:
cm_path = str(LOG_DIR / 'confusion_matrix.png')
plot_confusion_matrix(
    test_labels.astype(int), test_preds,
    output_path=cm_path,
    title='DiT — Ternary Confusion Matrix (Test Set)'
)
from IPython.display import Image as IPImage
IPImage(cm_path)

## 6 — ROC and PR Curves


In [ ]:
roc_path = str(LOG_DIR / 'roc_curve.png')
pr_path  = str(LOG_DIR / 'pr_curve.png')

plot_roc_curve(test_labels.astype(int), test_probs, output_path=roc_path, title='DiT — ROC Curve (Test Set)')
plot_pr_curve( test_labels.astype(int), test_probs, output_path=pr_path,  title='DiT — Precision-Recall Curve (Test Set)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, img_path, title in [
    (axes[0], roc_path, 'ROC Curve'),
    (axes[1], pr_path,  'Precision-Recall Curve'),
]:
    img = plt.imread(img_path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 7 — Per-Institution Analysis


In [ ]:
# ── Extract institutions ──────────────────────────────────────────────────────
if 'institution' in test_meta.columns:
    institutions = test_meta['institution'].values[:len(test_labels)]
else:
    institutions = np.array(['unknown'] * len(test_labels))
    print('Warning: institution column not found in metadata — using dummy.')

inst_metrics_df = compute_per_institution_metrics(
    test_labels.astype(int), test_probs, institutions, thresholds
)

print('Per-Institution Metrics:')
print(inst_metrics_df.to_string(index=False, float_format='{:.4f}'.format))

# ── Flag institutions above false-safe target ─────────────────────────────────
flagged = inst_metrics_df[inst_metrics_df['false_safe_rate'] > 0.05]
if len(flagged):
    print(f'\n⚠ Institutions EXCEEDING 5% false-safe target: {flagged["institution"].tolist()}')
else:
    print('\nAll institutions within 5% false-safe target.')

# ── Bar chart ─────────────────────────────────────────────────────────────────
bar_path = str(LOG_DIR / 'per_institution_false_safe.png')
plot_per_institution_metrics(inst_metrics_df, 'false_safe_rate', output_path=bar_path)
IPImage(bar_path)

## 8 — Error Analysis: False-Safe Cases


In [ ]:
# False-safe: truly risky (label=1) but predicted safe (pred=0)
false_safe_mask = (test_labels.astype(int) == 1) & (test_preds == 0)
print(f'Total false-safe cases: {false_safe_mask.sum()} / {len(test_labels)}')

false_safe_df = test_meta.iloc[:len(test_labels)][false_safe_mask].copy()

if len(false_safe_df) == 0:
    print('No false-safe cases — model is performing well on this metric!')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # 1. risk_score distribution
    if 'risk_score' in false_safe_df.columns:
        score_counts = false_safe_df['risk_score'].value_counts().sort_index()
        axes[0].bar(score_counts.index.astype(str), score_counts.values, color='tomato')
        axes[0].set_title('Risk Score Among False-Safes')
        axes[0].set_xlabel('risk_score')
        axes[0].set_ylabel('Count')
    else:
        axes[0].text(0.5, 0.5, 'risk_score\nnot available', ha='center', va='center', transform=axes[0].transAxes)
        axes[0].axis('off')

    # 2. Rubric component distribution (D/H/S/L)
    rubric_cols = [c for c in ['D', 'H', 'S', 'L'] if c in false_safe_df.columns]
    if rubric_cols:
        means = false_safe_df[rubric_cols].mean()
        axes[1].bar(means.index, means.values, color='darkorange')
        axes[1].set_title('Mean Rubric Components (False-Safes)')
        axes[1].set_ylabel('Mean Score')
        axes[1].set_ylim(0, 3)
    else:
        axes[1].text(0.5, 0.5, 'D/H/S/L\nnot available', ha='center', va='center', transform=axes[1].transAxes)
        axes[1].axis('off')

    # 3. Institution breakdown
    if 'institution' in false_safe_df.columns:
        inst_counts = false_safe_df['institution'].value_counts()
        axes[2].bar(range(len(inst_counts)), inst_counts.values, color='steelblue')
        axes[2].set_xticks(range(len(inst_counts)))
        axes[2].set_xticklabels(inst_counts.index, rotation=30, ha='right', fontsize=8)
        axes[2].set_title('False-Safes by Institution')
        axes[2].set_ylabel('Count')
    else:
        axes[2].text(0.5, 0.5, 'institution\nnot available', ha='center', va='center', transform=axes[2].transAxes)
        axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ── Show top false-safe samples ────────────────────────────────────────────
    display_cols = [c for c in ['page_id', 'institution', 'template_family', 'risk_score', 'D', 'H', 'S', 'L'] if c in false_safe_df.columns]
    if display_cols:
        print(f'\nSample false-safe records (first 20):')
        print(false_safe_df[display_cols].head(20).to_string(index=False))

## 9 — Score Distribution on Test Set


In [ ]:
dist_path = str(LOG_DIR / 'test_score_distribution.png')
plot_score_distribution(
    test_probs, test_labels.astype(int),
    output_path=dist_path
)
IPImage(dist_path)

## 10 — Production Readiness Checklist

Fill in the status column based on the metrics computed above.

| Criterion | Target | Status |
|-----------|--------|--------|
| Grouped test performance stable | Val ≈ Test F1 gap < 0.05 | [ ] |
| False-safe rate below target | ≤ 5% globally | [ ] |
| Calibration acceptable | ECE < 0.10 | [ ] |
| Borderline diagnostic set reviewed manually | risk_score 4–6 failure modes understood | [ ] |
| Per-institution failure spread understood | No institution > 10% false-safe | [ ] |

### How to use this checklist

1. **Grouped stability**: compare `val_metrics['f1']` from notebook 05 with `test_metrics['f1']` above. A gap > 0.05 suggests institution-level distribution shift.
2. **False-safe rate**: `test_metrics['false_safe_rate']` must be ≤ 0.05. If not, lower `T_low` (tighter safe threshold) and re-run.
3. **ECE**: `test_metrics['ece']` must be < 0.10. If not, re-calibrate with more val data or try label-smoothed training.
4. **Borderline set**: manually inspect the false-safe cases with `risk_score` 4–6 from the error analysis above. Common failure modes are: partial forms with enough template content to look safe, and mixed print/handwriting pages.
5. **Per-institution**: `inst_metrics_df` above shows institution-level breakdown. Flag any institution with `false_safe_rate > 0.10` for additional annotation.

Once all boxes are checked, proceed to **Phase 3 — Production Readiness** (packaging, threshold versioning, deployment).


In [ ]:
# ── Automated summary printout ────────────────────────────────────────────────
print('=' * 55)
print('PRODUCTION READINESS SUMMARY')
print('=' * 55)

checks = {
    'false_safe_rate <= 0.05': test_metrics['false_safe_rate'] <= 0.05,
    'ece < 0.10':              test_metrics['ece'] < 0.10,
    'roc_auc >= 0.80':         test_metrics['roc_auc'] >= 0.80,
    'review_rate <= 0.30':     test_metrics['review_rate'] <= 0.30,
}

all_pass = True
for criterion, passed in checks.items():
    status = 'PASS' if passed else 'FAIL'
    if not passed:
        all_pass = False
    print(f'  [{status}]  {criterion}')

print('-' * 55)
if all_pass:
    print('All automated checks PASSED. Proceed to Phase 3.')
else:
    print('Some checks FAILED. Address failures before deployment.')

# ── Save eval summary ─────────────────────────────────────────────────────────
import json
summary = {
    'test_metrics':  test_metrics,
    'thresholds':    thresholds,
    'temperature':   calibrator.temperature,
    'checks_passed': checks,
    'all_pass':      all_pass,
}
summary_path = LOG_DIR / 'eval_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=float)
print(f'\nEval summary saved → {summary_path}')